In [ ]:
"""
semantic_rubert.py
Семантический анализ с использованием ruBERT.
Три способа получения векторов для слов в контексте.
"""

from pathlib import Path

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

from filter_data_llama_6 import get_all_models
from semantic_utils_2 import (
    top_k_values, nn_thresholds,  # переименованные константы
    load_data, prep_model_df, run_pipeline,)

# Настройки
rubert = "DeepPavlov/rubert-base-cased"  # название модели
last_layer = 1 # берётся только последний (12-й) слой
output = Path("output/semantic_rubert")
output.mkdir(parents=True, exist_ok=True)

# определяем устройство (GPU если есть)
device = "cuda" if torch.cuda.is_available() else "cpu"


print(f"Загружаем ruBERT: {rubert}...")

tokenizer = AutoTokenizer.from_pretrained(rubert)

model = AutoModel.from_pretrained(rubert, output_hidden_states=True).to(device).eval()  # переводим в режим оценки (без dropout)

razzmer = model.config.hidden_size
print(f"  Размерность={razzmer}, слоев={model.config.num_hidden_layers}")



# Вспомогательные функции для работы с BERT

@torch.no_grad()  # отключаем вычисление градиентов (экономит память)
def _hidden(text: str) -> torch.Tensor:
    """
    Получаем скрытые состояния для текста.
    Усредняем по последним last_layer слоям.
    """
    # токенизируем текст
    enc = tokenizer(text, return_tensors="pt",
                    truncation=True, max_length=512).to(device)
    out = model(**enc)

    # берем последние N слоев и усредняем
    hs = torch.stack(out.hidden_states[-last_layer:]).mean(0)  # (1, T, dim)
    return hs.squeeze(0)  # убираем batch dimension -> (T, dim)


@torch.no_grad()
def cls_vec(text: str) -> np.ndarray:
    """Вектор [CLS] токена для текста (обычно используется для классификации)"""
    return _hidden(text)[0].cpu().numpy()  # [CLS] всегда первый токен


@torch.no_grad()
def word_isolated_vec(word: str) -> np.ndarray:
    """
    Вектор для слова без контекста.
    Берем среднее по всем субтокенам слова.
    """
    enc = tokenizer(word, return_tensors="pt",
                    truncation=True, max_length=64).to(device)
    out = model(**enc)
    hs = torch.stack(out.hidden_states[-last_layer:]).mean(0).squeeze(0)

    # пропускаем [CLS] (индекс 0) и [SEP] (последний)
    toks = hs[1:-1]
    if len(toks) == 0:
        return np.zeros(razzmer)
    return toks.mean(0).cpu().numpy()


@torch.no_grad()
def target_token_vec(left_ctx: str, word: str) -> np.ndarray:
    """
    Способ A: подставляем слово в контекст и берем среднее по его субтокенам.
    """
    text = left_ctx.rstrip() + " " + word
    enc = tokenizer(text, return_tensors="pt",
                    truncation=True, max_length=512).to(device)
    ids_list = enc["input_ids"][0].tolist()

    # токены слова без специальных токенов
    word_ids_raw = tokenizer(word, add_special_tokens=False)["input_ids"]
    n = len(word_ids_raw)

    # ищем позиции субтокенов слова (с конца, чтобы найти последнее вхождение)
    word_pos = []
    for i in range(len(ids_list) - n, -1, -1):
        if ids_list[i:i + n] == word_ids_raw:
            word_pos = list(range(i, i + n))
            break

    if not word_pos:
        return np.zeros(razzmer)  # если не нашли - возвращаем нули

    # берем средний вектор по субтокенам
    hs = _hidden(text)
    return hs[word_pos].mean(0).cpu().numpy()


# Кэширование результатов (чтобы не вычислять дважды одно и то же)

_cache_cls: dict[str, np.ndarray] = {}  # кэш для [CLS] векторов
_cache_isolated: dict[str, np.ndarray] = {}  # кэш для изолированных слов
_cache_A: dict[tuple, np.ndarray] = {}  # кэш для способа A


def cached_cls(ctx: str) -> np.ndarray:
    """Возвращает [CLS] вектор из кэша или вычисляет заново"""
    if ctx not in _cache_cls:
        _cache_cls[ctx] = cls_vec(ctx)
    return _cache_cls[ctx]


def cached_isolated(word: str) -> np.ndarray:
    """Возвращает вектор изолированного слова из кэша"""
    if word not in _cache_isolated:
        _cache_isolated[word] = word_isolated_vec(word)
    return _cache_isolated[word]


def cached_A(ctx: str, word: str) -> np.ndarray:
    """Возвращает вектор способа A из кэша"""
    key = (ctx, word)
    if key not in _cache_A:
        _cache_A[key] = target_token_vec(ctx, word)
    return _cache_A[key]


# Три способа получения векторов

def get_vec_A(lemma: str, upos: str, left_ctx: str = "") -> np.ndarray:
    """Способ A: вектор слова в контексте"""
    return cached_A(left_ctx, lemma)


def get_vec_B(lemma: str, upos: str, left_ctx: str = "") -> np.ndarray:
    """Способ B: усредняем [CLS] контекста и изолированный вектор слова"""
    v_ctx = cached_cls(left_ctx)  # вектор контекста
    v_word = cached_isolated(lemma)  # вектор слова без контекста
    return (v_ctx + v_word) / 2.0


def get_vec_C(lemma: str, upos: str, left_ctx: str = "") -> np.ndarray:
    """Способ C: residual = v_A - CLS_контекста (убираем фоновый сигнал)"""
    v_a = cached_A(left_ctx, lemma)  # вектор слова в контексте
    v_ctx = cached_cls(left_ctx)  # вектор контекста
    return v_a - v_ctx




# Загружаем данные и запускаем

print("\nЗагружаем данные...")
human_deduped, context_lookup = load_data()

# словарь методов: имя -> функция получения вектора
methods = {
    "rubert_A": get_vec_A,
    "rubert_B": get_vec_B,
    "rubert_C": get_vec_C,
}

# запускаем для каждой языковой модели
for model_name, model_df in get_all_models().items():
    print(f"\n")
    print(f" model: {model_name}")

    model_slug = model_name.replace(" ", "_").replace("-", "_")
    model_deduped = prep_model_df(model_df)

    # для каждого метода считаем метрики
    for method_name, get_vec_fn in methods.items():
        print(f"\n метод сейчас {method_name}")

        # запускаем пайплайн
        df = run_pipeline(
            approach_name=method_name,
            get_vec=get_vec_fn,
            human_deduped=human_deduped,
            model_deduped=model_deduped,
            context_lookup=context_lookup,
            out_dir=output,
            model_slug=model_slug,
        )

        # считаем средние метрики по top-k
        metric_cols = ["centroid_sim"] + [
            f"{m}_{t}"
            for t in [70, 80, 90]  # пороги
            for m in ("recall", "precision", "f1")
        ]
        available = [c for c in metric_cols if c in df.columns]
        summary = df.groupby("top_k")[available].mean().round(4).reset_index()

        # сохраняем
        summary.to_csv(
            output / f"summary_{method_name}_{model_slug}.csv", index=False
        )

        print(f"\n Сводка {model_name} / {method_name}:")
        print("  " + summary.to_string(index=False))

    # выводим информацию о кэше
    print(f"\nCLS={len(_cache_cls)}, "
          f"isolated={len(_cache_isolated)}, A={len(_cache_A)}")

print(f"законили {output}")

Загружаем ruBERT: DeepPavlov/rubert-base-cased...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Размерность=768, слоев=12

Загружаем данные...
Загружено людей: 2659 записей, 144 разных контекстов


 model: GPT-4o-mini

 метод сейчас rubert_A
  [rubert_A] Найдено общих контекстов: 144
    Обработано 20/144...
    Обработано 40/144...
    Обработано 60/144...
    Обработано 80/144...
    Обработано 100/144...
    Обработано 120/144...
    Обработано 140/144...
  Сохранено в per_context_rubert_A_GPT_4o_mini.csv (864 строк)

 Сводка GPT-4o-mini / rubert_A:
   top_k  centroid_sim  recall_70  precision_70  f1_70  recall_80  precision_80  f1_80  recall_90  precision_90  f1_90
     1        0.8205     0.4264        0.9028 0.5177     0.3240        0.8542 0.4102     0.2535        0.7778 0.3274
     5        0.9142     0.7198        0.9015 0.7737     0.5542        0.8029 0.6235     0.4541        0.7119 0.5188
    10        0.9199     0.7600        0.8933 0.7976     0.5946        0.7908 0.6501     0.4937        0.6965 0.5427
    20        0.9209     0.7838        0.8917 0.8113     0.6214  